# 05 — Band Importance

Estimates the importance of each input channel using two complementary methods:

1. **Grad × Input** — fast gradient-based saliency, averaged over the val set
2. **Per-channel occlusion** — zero out one channel at a time, measure accuracy drop

Channels are ordered as: `[ref_band1..8, query_band1..8, ref_mask, query_mask]`
where band order is: coastal_blue, blue, green_i, green_ii, yellow, red, rededge, nir.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH    = "/path/to/model_states/best.pt"
PROJECT_DIR  = "/path/to/ct_classifier"
FIGURES_DIR  = "figures"

N_BATCHES_GRAD = 25   # batches to average grad×input over (fast)
N_BATCHES_OCCL = 6    # batches for occlusion (slower); set 0 to skip
TARGET_CLASS   = "pred"  # "pred" uses each sample's predicted class; or int e.g. 1
SEED           = 0

In [ ]:
import os, sys, random
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.dataset import BleachDataset
from ct_classifier.model import CustomResNet

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()

dl_val = DataLoader(
    BleachDataset(cfg, split="val", eval=True),
    batch_size=cfg.get("batch_size", 32),
    shuffle=True,
    num_workers=cfg.get("num_workers", 0),
)

# Channel labels: 8 ref bands + 8 query bands + 2 masks
BAND_NAMES = ["coastal_blue", "blue", "green_i", "green_ii",
              "yellow", "red", "rededge", "nir"]
CHANNEL_LABELS = (
    [f"ref_{b}"   for b in BAND_NAMES] +
    [f"qry_{b}"   for b in BAND_NAMES] +
    ["ref_mask", "qry_mask"]
)

In [ ]:
# ── Method 1: Grad × Input ────────────────────────────────────────────────────
grad_input_sum = np.zeros(18, dtype=np.float64)
n_samples = 0

for batch_idx, (data, labels, _) in enumerate(dl_val):
    if batch_idx >= N_BATCHES_GRAD:
        break

    data = data.to(device).requires_grad_(True)
    labels = labels.to(device).long()

    logits = model(data)

    if TARGET_CLASS == "pred":
        target_idx = logits.argmax(dim=1)
    else:
        target_idx = torch.full((len(data),), int(TARGET_CLASS),
                                dtype=torch.long, device=device)

    score = logits.gather(1, target_idx.unsqueeze(1)).squeeze(1).sum()
    model.zero_grad()
    score.backward()

    saliency = (data.grad * data).abs()   # [B, 18, H, W]
    grad_input_sum += saliency.mean(dim=(0, 2, 3)).detach().cpu().numpy()
    n_samples += 1

grad_importance = grad_input_sum / n_samples
grad_importance /= grad_importance.sum() + 1e-8
print("Grad×Input done.")

In [ ]:
# ── Method 2: Per-channel occlusion ──────────────────────────────────────────
occl_importance = np.zeros(18, dtype=np.float64)

if N_BATCHES_OCCL > 0:
    baseline_correct = 0
    channel_correct  = np.zeros(18, dtype=np.int64)
    n_total = 0

    with torch.no_grad():
        for batch_idx, (data, labels, _) in enumerate(dl_val):
            if batch_idx >= N_BATCHES_OCCL:
                break

            data   = data.to(device)
            labels = labels.to(device).long()

            base_preds = model(data).argmax(dim=1)
            baseline_correct += (base_preds == labels).sum().item()
            n_total += len(labels)

            for ch in range(18):
                x_occ = data.clone()
                x_occ[:, ch, :, :] = 0.0
                occ_preds = model(x_occ).argmax(dim=1)
                channel_correct[ch] += (occ_preds == labels).sum().item()

    base_acc = baseline_correct / n_total
    occ_accs = channel_correct / n_total
    occl_importance = base_acc - occ_accs   # positive = channel mattered
    print(f"Occlusion done. Baseline acc: {base_acc*100:.2f}%")
else:
    print("Occlusion skipped (N_BATCHES_OCCL=0).")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
x = np.arange(18)
colors = ["#2196F3"] * 8 + ["#FF5722"] * 8 + ["#9E9E9E"] * 2  # ref=blue, qry=orange, mask=grey

n_plots = 1 + (N_BATCHES_OCCL > 0)
fig, axes = plt.subplots(n_plots, 1, figsize=(12, 4 * n_plots))
if n_plots == 1:
    axes = [axes]

axes[0].bar(x, grad_importance, color=colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(CHANNEL_LABELS, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Normalized importance")
axes[0].set_title("Band importance — Grad × Input")

# legend
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color="#2196F3", label="Reference bands"),
    Patch(color="#FF5722", label="Query bands"),
    Patch(color="#9E9E9E", label="Masks"),
])

if N_BATCHES_OCCL > 0:
    axes[1].bar(x, occl_importance, color=colors)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(CHANNEL_LABELS, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylabel("Accuracy drop")
    axes[1].set_title("Band importance — Per-channel occlusion")
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "band_importance.png"), dpi=150, bbox_inches="tight")
plt.show()